# setting up the environment

In [ ]:
uv venv --python 3.11 .venv_rakuten

source .venv_rakuten/bin/activate
uv pip install --upgrade pip

uv pip install pandas 
uv pip install nltk
uv pip install bs4
uv pip install requests

python src/data/import_raw_data.py
# then download data from https://challengedata.ens.fr/participants/challenges/35/
# I think this needs to lead to raw/image_test and raw/image_train for the whole thing to work properly. 
# then: 
uv pip install dotenv
python src/data/make_dataset.py data/raw data/preprocessed

uv pip install tensorflow
uv pip install scikit-learn
uv pip install pillow

# in train_moedl.py, line 203 I commented out "new_y_train = new_y_train.values.reshape(1350).astype("int")" because of 
# "ValueError: cannot reshape array of size 2700 into shape (1350,)"
# I think this was because of dataframe/array coversion issues, but it seemed to work without it anyways

python src/main.py
 # import pickle

In [13]:
#from features.build_features import DataImporter, TextPreprocessor, ImagePreprocessor
#from models.train_model import TextLSTMModel, ImageVGG16Model, concatenate
from tensorflow import keras
import pickle
import tensorflow as tf


data_importer = DataImporter()
df = data_importer.load_data()
X_train, X_val, _, y_train, y_val, _ = data_importer.split_train_test(df)

# Preprocess text and images
text_preprocessor = TextPreprocessor()
image_preprocessor = ImagePreprocessor()
text_preprocessor.preprocess_text_in_df(X_train, columns=["description"])
text_preprocessor.preprocess_text_in_df(X_val, columns=["description"])
image_preprocessor.preprocess_images_in_df(X_train)
image_preprocessor.preprocess_images_in_df(X_val)


[nltk_data] Downloading package punkt to /usr/local/lib/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /usr/local/lib/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/local/lib/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [14]:

# Train LSTM model
print("Training LSTM Model")
text_lstm_model = TextLSTMModel()
text_lstm_model.preprocess_and_fit(X_train, y_train, X_val, y_val)
print("Finished training LSTM")


Training LSTM Model
505/507 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.2907 - loss: 2.4938

507/507 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.4391 - loss: 1.9231 - val_accuracy: 0.6311 - val_loss: 1.2297
Finished training LSTM


In [16]:

print("Training VGG")
# Train VGG16 model
image_vgg16_model = ImageVGG16Model()
image_vgg16_model.preprocess_and_fit(X_train, y_train, X_val, y_val)
print("Finished training VGG")

Training VGG
Found 16200 validated image filenames belonging to 27 classes.
Found 1350 validated image filenames belonging to 27 classes.
507/507 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.1571 - loss: 7.9985

507/507 ━━━━━━━━━━━━━━━━━━━━ 4037s 8s/step - accuracy: 0.1821 - loss: 4.1424 - val_accuracy: 0.2126 - val_loss: 2.8881
Finished training VGG


In [2]:
# features
import pandas as pd
import nltk
from bs4 import BeautifulSoup
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import pickle
import math


class DataImporter:
    def __init__(self, filepath="data/preprocessed"):
        self.filepath = filepath

    def load_data(self):
        data = pd.read_csv(f"{self.filepath}/X_train_update.csv")
        data["description"] = data["designation"] + str(data["description"])
        data = data.drop(["Unnamed: 0", "designation"], axis=1)

        target = pd.read_csv(f"{self.filepath}/Y_train_CVw08PX.csv")
        target = target.drop(["Unnamed: 0"], axis=1)
        modalite_mapping = {
            modalite: i for i, modalite in enumerate(target["prdtypecode"].unique())
        }
        target["prdtypecode"] = target["prdtypecode"].replace(modalite_mapping)

        with open("models/mapper.pkl", "wb") as fichier:
            pickle.dump(modalite_mapping, fichier)

        df = pd.concat([data, target], axis=1)

        return df

    def split_train_test(self, df, samples_per_class=600):

        grouped_data = df.groupby("prdtypecode")

        X_train_samples = []
        X_test_samples = []

        for _, group in grouped_data:
            samples = group.sample(n=samples_per_class, random_state=42)
            X_train_samples.append(samples)

            remaining_samples = group.drop(samples.index)
            X_test_samples.append(remaining_samples)

        X_train = pd.concat(X_train_samples)
        X_test = pd.concat(X_test_samples)

        X_train = X_train.sample(frac=1, random_state=42).reset_index(drop=True)
        X_test = X_test.sample(frac=1, random_state=42).reset_index(drop=True)

        y_train = X_train["prdtypecode"]
        X_train = X_train.drop(["prdtypecode"], axis=1)

        y_test = X_test["prdtypecode"]
        X_test = X_test.drop(["prdtypecode"], axis=1)

        val_samples_per_class = 50

        grouped_data_test = pd.concat([X_test, y_test], axis=1).groupby("prdtypecode")

        X_val_samples = []
        y_val_samples = []

        for _, group in grouped_data_test:
            samples = group.sample(n=val_samples_per_class, random_state=42)
            X_val_samples.append(samples[["description", "productid", "imageid"]])
            y_val_samples.append(samples["prdtypecode"])

        X_val = pd.concat(X_val_samples)
        y_val = pd.concat(y_val_samples)

        X_val = X_val.sample(frac=1, random_state=42).reset_index(drop=True)
        y_val = y_val.sample(frac=1, random_state=42).reset_index(drop=True)

        return X_train, X_val, X_test, y_train, y_val, y_test


class ImagePreprocessor:
    def __init__(self, filepath="data/preprocessed/image_train"):
        self.filepath = filepath

    def preprocess_images_in_df(self, df):
        df["image_path"] = (
            f"{self.filepath}/image_"
            + df["imageid"].astype(str)
            + "_product_"
            + df["productid"].astype(str)
            + ".jpg"
        )


class TextPreprocessor:
    def __init__(self):
        nltk.download("punkt")
        nltk.download("stopwords")
        nltk.download("wordnet")
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(
            stopwords.words("french")
        )  # Vous pouvez choisir une autre langue si nécessaire

    def preprocess_text(self, text):

        if isinstance(text, float) and math.isnan(text):
            return ""
        # Supprimer les balises HTML
        text = BeautifulSoup(text, "html.parser").get_text()

        # Supprimer les caractères non alphabétiques
        text = re.sub(r"[^a-zA-Z]", " ", text)

        # Tokenization
        words = word_tokenize(text.lower())

        # Suppression des stopwords et lemmatisation
        filtered_words = [
            self.lemmatizer.lemmatize(word)
            for word in words
            if word not in self.stop_words
        ]

        return " ".join(filtered_words[:10])

    def preprocess_text_in_df(self, df, columns):
        for column in columns:
            df[column] = df[column].apply(self.preprocess_text)


In [3]:
# models
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.layers import Input, Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import pandas as pd
from sklearn.utils import resample
import numpy as np
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from sklearn.metrics import accuracy_score
from tensorflow import keras
import pickle
import json


class TextLSTMModel:
    def __init__(self, max_words=10000, max_sequence_length=10):
        self.max_words = max_words
        self.max_sequence_length = max_sequence_length
        self.tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
        self.model = None

    def preprocess_and_fit(self, X_train, y_train, X_val, y_val):
        self.tokenizer.fit_on_texts(X_train["description"])

        tokenizer_config = self.tokenizer.to_json()
        with open("models/tokenizer_config.json", "w", encoding="utf-8") as json_file:
            json_file.write(tokenizer_config)

        train_sequences = self.tokenizer.texts_to_sequences(X_train["description"])
        train_padded_sequences = pad_sequences(
            train_sequences,
            maxlen=self.max_sequence_length,
            padding="post",
            truncating="post",
        )

        val_sequences = self.tokenizer.texts_to_sequences(X_val["description"])
        val_padded_sequences = pad_sequences(
            val_sequences,
            maxlen=self.max_sequence_length,
            padding="post",
            truncating="post",
        )

        text_input = Input(shape=(self.max_sequence_length,))
        embedding_layer = Embedding(input_dim=self.max_words, output_dim=128)(
            text_input
        )
        lstm_layer = LSTM(128)(embedding_layer)
        output = Dense(27, activation="softmax")(lstm_layer)

        self.model = Model(inputs=[text_input], outputs=output)

        self.model.compile(
            optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
        )

        lstm_callbacks = [
            ModelCheckpoint(
                filepath="models/best_lstm_model.h5", save_best_only=True
            ),  # Enregistre le meilleur modèle
            EarlyStopping(
                patience=3, restore_best_weights=True
            ),  # Arrête l'entraînement si la performance ne s'améliore pas
            TensorBoard(log_dir="logs"),  # Enregistre les journaux pour TensorBoard
        ]

        self.model.fit(
            [train_padded_sequences],
            tf.keras.utils.to_categorical(y_train, num_classes=27),
            epochs=1,
            batch_size=32,
            validation_data=(
                [val_padded_sequences],
                tf.keras.utils.to_categorical(y_val, num_classes=27),
            ),
            callbacks=lstm_callbacks,
        )


class ImageVGG16Model:
    def __init__(self):
        self.model = None

    def preprocess_and_fit(self, X_train, y_train, X_val, y_val):
        # Paramètres
        batch_size = 32
        num_classes = 27

        df_train = pd.concat([X_train, y_train.astype(str)], axis=1)
        df_val = pd.concat([X_val, y_val.astype(str)], axis=1)

        # Créer un générateur d'images pour le set d'entraînement
        train_datagen = ImageDataGenerator()  # Normalisation des valeurs de pixel
        train_generator = train_datagen.flow_from_dataframe(
            dataframe=df_train,
            x_col="image_path",
            y_col="prdtypecode",
            target_size=(224, 224),  # Adapter à la taille d'entrée de VGG16
            batch_size=batch_size,
            class_mode="categorical",  # Utilisez 'categorical' pour les entiers encodés en one-hot
            shuffle=True,
        )

        # Créer un générateur d'images pour le set de validation
        val_datagen = ImageDataGenerator()  # Normalisation des valeurs de pixel
        val_generator = val_datagen.flow_from_dataframe(
            dataframe=df_val,
            x_col="image_path",
            y_col="prdtypecode",
            target_size=(224, 224),
            batch_size=batch_size,
            class_mode="categorical",
            shuffle=False,  # Pas de mélange pour le set de validation
        )

        image_input = Input(
            shape=(224, 224, 3)
        )  # Adjust input shape according to your images

        vgg16_base = VGG16(
            include_top=False, weights="imagenet", input_tensor=image_input
        )

        x = vgg16_base.output
        x = Flatten()(x)
        x = Dense(256, activation="relu")(x)  # Add some additional layers if needed
        output = Dense(num_classes, activation="softmax")(x)

        self.model = Model(inputs=vgg16_base.input, outputs=output)

        for layer in vgg16_base.layers:
            layer.trainable = False

        self.model.compile(
            optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
        )

        vgg_callbacks = [
            ModelCheckpoint(
                filepath="models/best_vgg16_model.h5", save_best_only=True
            ),  # Enregistre le meilleur modèle
            EarlyStopping(
                patience=3, restore_best_weights=True
            ),  # Arrête l'entraînement si la performance ne s'améliore pas
            TensorBoard(log_dir="logs"),  # Enregistre les journaux pour TensorBoard
        ]

        self.model.fit(
            train_generator,
            epochs=1,
            validation_data=val_generator,
            callbacks=vgg_callbacks,
        )


class concatenate:
    def __init__(self, tokenizer, lstm, vgg16):
        self.tokenizer = tokenizer
        self.lstm = lstm
        self.vgg16 = vgg16

    def preprocess_image(self, image_path, target_size):
        img = load_img(image_path, target_size=target_size)
        img_array = img_to_array(img)
        img_array = preprocess_input(img_array)
        return img_array

    def predict(
        self, X_train, y_train, new_samples_per_class=50, max_sequence_length=10
    ):
        num_classes = 27

        new_X_train = pd.DataFrame(columns=X_train.columns)
        new_y_train = pd.DataFrame(
            columns=[0]
        )  # Créez la structure pour les étiquettes

        # Boucle à travers chaque classe
        for class_label in range(num_classes):
            # Indices des échantillons appartenant à la classe actuelle
            indices = np.where(y_train == class_label)[0]

            # Sous-échantillonnage aléatoire pour sélectionner 'new_samples_per_class' échantillons
            sampled_indices = resample(
                indices, n_samples=new_samples_per_class, replace=False, random_state=42
            )

            # Ajout des échantillons sous-échantillonnés et de leurs étiquettes aux DataFrames
            new_X_train = pd.concat([new_X_train, X_train.loc[sampled_indices]])
            new_y_train = pd.concat([new_y_train, y_train.loc[sampled_indices]])

        # Réinitialiser les index des DataFrames
        new_X_train = new_X_train.reset_index(drop=True)
        new_y_train = new_y_train.reset_index(drop=True)
        new_y_train = new_y_train.values.reshape(2700).astype("int")

        # Charger les modèles préalablement sauvegardés
        tokenizer = self.tokenizer
        lstm_model = self.lstm
        vgg16_model = self.vgg16

        train_sequences = tokenizer.texts_to_sequences(new_X_train["description"])
        train_padded_sequences = pad_sequences(
            train_sequences, maxlen=10, padding="post", truncating="post"
        )

        # Paramètres pour le prétraitement des images
        target_size = (
            224,
            224,
            3,
        )  # Taille cible pour le modèle VGG16, ajustez selon vos besoins

        images_train = new_X_train["image_path"].apply(
            lambda x: self.preprocess_image(x, target_size)
        )

        images_train = tf.convert_to_tensor(images_train.tolist(), dtype=tf.float32)

        lstm_proba = lstm_model.predict([train_padded_sequences])

        vgg16_proba = vgg16_model.predict([images_train])

        return lstm_proba, vgg16_proba, new_y_train

    def optimize(self, lstm_proba, vgg16_proba, y_train):
        # Recherche des poids optimaux en utilisant la validation croisée
        best_weights = None
        best_accuracy = 0.0

        for lstm_weight in np.linspace(0, 1, 101):  # Essayer différents poids pour LSTM
            vgg16_weight = 1.0 - lstm_weight  # Le poids total doit être égal à 1

            combined_predictions = (lstm_weight * lstm_proba) + (
                vgg16_weight * vgg16_proba
            )
            final_predictions = np.argmax(combined_predictions, axis=1)
            accuracy = accuracy_score(y_train, final_predictions)

            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_weights = (lstm_weight, vgg16_weight)

        return best_weights


2026-02-04 11:39:18.206827: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [6]:
#from features.build_features import DataImporter, TextPreprocessor, ImagePreprocessor
#from models.train_model import TextLSTMModel, ImageVGG16Model, concatenate
from tensorflow import keras
import pickle
import tensorflow as tf
data_importer = DataImporter()
df = data_importer.load_data()
X_train, X_val, _, y_train, y_val, _ = data_importer.split_train_test(df)


# Preprocess text and images
text_preprocessor = TextPreprocessor()
image_preprocessor = ImagePreprocessor()
text_preprocessor.preprocess_text_in_df(X_train, columns=["description"])
text_preprocessor.preprocess_text_in_df(X_val, columns=["description"])
image_preprocessor.preprocess_images_in_df(X_train)
image_preprocessor.preprocess_images_in_df(X_val)



[nltk_data] Downloading package punkt to /usr/local/lib/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /usr/local/lib/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/local/lib/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [7]:

# Train LSTM model
print("Training LSTM Model")
text_lstm_model = TextLSTMModel()
text_lstm_model.preprocess_and_fit(X_train, y_train, X_val, y_val)
print("Finished training LSTM")


Training LSTM Model
506/507 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.2888 - loss: 2.4989

507/507 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.4323 - loss: 1.9505 - val_accuracy: 0.6333 - val_loss: 1.2635
Finished training LSTM


In [8]:

print("Training VGG")
# Train VGG16 model
image_vgg16_model = ImageVGG16Model()
image_vgg16_model.preprocess_and_fit(X_train, y_train, X_val, y_val)
print("Finished training VGG")


Training VGG
Found 16200 validated image filenames belonging to 27 classes.
Found 1350 validated image filenames belonging to 27 classes.
507/507 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.1344 - loss: 9.9362

507/507 ━━━━━━━━━━━━━━━━━━━━ 2666s 5s/step - accuracy: 0.1513 - loss: 4.7271 - val_accuracy: 0.1985 - val_loss: 2.9186
Finished training VGG


In [37]:
new_samples_per_class=50
max_sequence_length=10

num_classes = 27

new_X_train = pd.DataFrame(columns=X_train.columns)
new_y_train = pd.DataFrame(
    columns=[0]
)  # Créez la structure pour les étiquettes

# Boucle à travers chaque classe
for class_label in range(num_classes):
    # Indices des échantillons appartenant à la classe actuelle
    indices = np.where(y_train == class_label)[0]

    # Sous-échantillonnage aléatoire pour sélectionner 'new_samples_per_class' échantillons
    sampled_indices = resample(
        indices, n_samples=new_samples_per_class, replace=False, random_state=42
    )

    # Ajout des échantillons sous-échantillonnés et de leurs étiquettes aux DataFrames
    new_X_train = pd.concat([new_X_train, X_train.loc[sampled_indices]])
    new_y_train = pd.concat([new_y_train, y_train.loc[sampled_indices]])

# Réinitialiser les index des DataFrames
new_X_train = new_X_train.reset_index(drop=True)
new_y_train = new_y_train.reset_index(drop=True)

In [39]:
new_y_train

,0,prdtypecode
0,NaN,0.0
1,NaN,0.0
2,NaN,0.0
3,NaN,0.0
4,NaN,0.0
...,...,...
1345,NaN,26.0
1346,NaN,26.0
1347,NaN,26.0
1348,NaN,26.0


In [36]:
new_X_train#new_y_train.prdtypecode.reshape(1350).astype("int")

,description,productid,imageid,image_path
2653,couleurs noir biographie genre nan nan pilot s...,4693593,981149525,data/preprocessed/image_train/image_981149525_...
10981,year of research rent seeking nan nan pilot st...,74519004,907246493,data/preprocessed/image_train/image_907246493_...
15034,physik neu bayern nan nan pilot style touch pe...,87117625,932538725,data/preprocessed/image_train/image_932538725_...
1915,the trouble with jack nan nan pilot style touc...,110186808,1077255854,data/preprocessed/image_train/image_1077255854...
4471,cause italienne eveques catholiques apologie p...,108592287,875676202,data/preprocessed/image_train/image_875676202_...
...,...,...,...,...
4849,jeu billes boule billard pool suprapro mm bce ...,141190582,893310883,data/preprocessed/image_train/image_893310883_...
13190,lovely baby kid b b fille gar ons cute cartoon,3898727002,1261422007,data/preprocessed/image_train/image_1261422007...
2949,nouveau b b stripe skinny pantalon mignon chau...,3898727707,1261424569,data/preprocessed/image_train/image_1261424569...
1665,jeu air hockey bureau pile enfant coussin air ...,522642853,1020940427,data/preprocessed/image_train/image_1020940427...


In [23]:

# Réinitialiser les index des DataFrames
new_X_train = new_X_train.reset_index(drop=True)
new_y_train = new_y_train.reset_index(drop=True)
new_y_train = new_y_train.values.reshape(2700).astype("int")


ValueError: cannot convert float NaN to integer

In [41]:

# Charger les modèles préalablement sauvegardés
tokenizer = model_concatenate.tokenizer


In [43]:
lstm_model = model_concatenate.lstm
vgg16_model = model_concatenate.vgg16

train_sequences = tokenizer.texts_to_sequences(new_X_train["description"])
train_padded_sequences = pad_sequences(
    train_sequences, maxlen=10, padding="post", truncating="post"
)

# Paramètres pour le prétraitement des images
target_size = (
    224,
    224,
    3,
)  # Taille cible pour le modèle VGG16, ajustez selon vos besoins

images_train = new_X_train["image_path"].apply(
    lambda x: model_concatenate.preprocess_image(x, target_size)
)

images_train = tf.convert_to_tensor(images_train.tolist(), dtype=tf.float32)

lstm_proba = lstm_model.predict([train_padded_sequences])

vgg16_proba = vgg16_model.predict([images_train])

#return lstm_proba, vgg16_proba, new_y_train

43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
43/43 ━━━━━━━━━━━━━━━━━━━━ 222s 5s/step


In [ ]:
lstm_proba, vgg16_proba, new_y_train

In [10]:

with open("models/tokenizer_config.json", "r", encoding="utf-8") as json_file:
    tokenizer_config = json_file.read()
tokenizer = tf.keras.preprocessing.text.tokenizer_from_json(tokenizer_config)
lstm = keras.models.load_model("models/best_lstm_model.h5")
vgg16 = keras.models.load_model("models/best_vgg16_model.h5")

print("Training the concatenate model")
model_concatenate = concatenate(tokenizer, lstm, vgg16)
lstm_proba, vgg16_proba, new_y_train = model_concatenate.predict(X_train, y_train)
best_weights = model_concatenate.optimize(lstm_proba, vgg16_proba, new_y_train)
print("Finished training concatenate model")


Training the concatenate model


ValueError: cannot convert float NaN to integer

In [ ]:

with open("models/best_weights.pkl", "wb") as file:
    pickle.dump(best_weights, file)

num_classes = 27

proba_lstm = keras.layers.Input(shape=(num_classes,))
proba_vgg16 = keras.layers.Input(shape=(num_classes,))

weighted_proba = keras.layers.Lambda(
    lambda x: best_weights[0] * x[0] + best_weights[1] * x[1]
)([proba_lstm, proba_vgg16])

concatenate_model = keras.models.Model(
    inputs=[proba_lstm, proba_vgg16], outputs=weighted_proba
)

# Enregistrer le modèle au format h5
concatenate_model.save("models/concatenate.h5")
